# Toxic Gaming Chat Classifier - Colab Training

**By Alex You**

This notebook uses Colab as a GPU runtime while keeping the project code in Python modules.

## 1. Enable GPU

In Colab, choose `Runtime > Change runtime type > T4 GPU`, then run this cell.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

CUDA available: True
GPU: Tesla T4
Fri Jul 17 00:18:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

## 2. Mount Google Drive

Set `PROJECT_DIR` to the folder that contains this repository in Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your repo is stored somewhere else in Drive.
PROJECT_DIR = '/content/drive/MyDrive/APS360/Final_Project'
%cd $PROJECT_DIR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/APS360/Final_Project


## 3. Install Dependencies

In [3]:
%pip install -r requirements.txt

Ignoring pywin32: markers 'platform_system == "Windows"' don't match your environment


## 4. Verify Or Prepare Expert-Labelled Data

Download L2DTnH's `2_16000_chatlogs_english_only.csv` to `data/l2dtnh/l2dtnh_english.csv`. The preparation script normalizes the messages, removes contradictory normalized labels, and creates match-grouped train/validation/test splits.

In [4]:
from pathlib import Path
import subprocess

import pandas as pd

raw_path = Path('data/l2dtnh/l2dtnh_english.csv')
prepared_path = Path('data/l2dtnh/l2dtnh_prepared.csv')

if not raw_path.exists():
    raise FileNotFoundError(
        'Upload 2_16000_chatlogs_english_only.csv as data/l2dtnh/l2dtnh_english.csv.'
    )

# Always rebuild so grouped split assignments and the audit match the current code.
subprocess.run(['python', 'prepare_l2dtnh.py'], check=True)

df = pd.read_csv(prepared_path)
print(df.head())
print(pd.crosstab(df['split'], df['label']))
print('Groups per split:', df.groupby('split')['group_id'].nunique().to_dict())

                                          raw_text  \
0                            olaf is bronze hahaha   
1  yes i know kayle is too but u gyus are premades   
2                                            kayle   
3                                if u dont die top   
4                    kayle plays like a challenger   

                                              text  label  group_id split  
0                            olaf is bronze hahaha      1      1009  test  
1  yes i know kayle is too but u gyus are premades      1      1009  test  
2                                            kayle      0      1009  test  
3                                if u dont die top      0      1009  test  
4                    kayle plays like a challenger      0      1009  test  
label     0    1
split           
test   1939  195
train  9123  927
val    1911  198
Groups per split: {'test': 16, 'train': 70, 'val': 15}


## 5. Run Validation-Only Screen, Three Seeds, Then Frozen Test

The experiment runner first screens configurations using validation data only, freezes the winner by mean validation toxic-class F1 across seeds 42, 43, and 44, and only then opens the grouped test split.

In [5]:
!python run_lstm_experiments.py --device cuda

Baseline validation: balanced accuracy 0.826; toxic-class F1 0.629
Run: corrected_current_seed42; device: cuda; test enabled: False
Epoch 01/15 | train loss 1.0393 f1@.50 0.326 | val loss 0.7958 p 0.711 r 0.510 f1 0.594 @ 0.80 <- saved
Epoch 02/15 | train loss 0.6860 f1@.50 0.486 | val loss 0.7050 p 0.608 r 0.641 f1 0.624 @ 0.80 <- saved
Epoch 03/15 | train loss 0.4663 f1@.50 0.602 | val loss 0.6859 p 0.657 r 0.571 f1 0.611 @ 0.90
Epoch 04/15 | train loss 0.3303 f1@.50 0.691 | val loss 0.7710 p 0.602 r 0.672 f1 0.635 @ 0.75 <- saved
Epoch 05/15 | train loss 0.2284 f1@.50 0.775 | val loss 0.9985 p 0.730 r 0.601 f1 0.659 @ 0.95 <- saved
Epoch 06/15 | train loss 0.2169 f1@.50 0.768 | val loss 1.1192 p 0.678 r 0.606 f1 0.640 @ 0.90
Epoch 07/15 | train loss 0.1235 f1@.50 0.857 | val loss 1.4498 p 0.787 r 0.540 f1 0.641 @ 0.95
Epoch 08/15 | train loss 0.0838 f1@.50 0.912 | val loss 1.3771 p 0.667 r 0.646 f1 0.656 @ 0.85
Early stopping after 8 epochs.
Run: weight_3_seed42; device: cuda; test 

## 6. Review Frozen Comparison

In [6]:
import json
from pathlib import Path

summary = json.loads(Path('artifacts/experiment_summary.json').read_text())
print('Frozen winner:', summary['winner'])
print('Baseline test F1:', summary['baseline_test']['f1'])
print('LSTM three-seed test F1:', summary['lstm_test_aggregate']['f1'])

Frozen winner: weight_7
Baseline test F1: 0.6547884187082406
LSTM three-seed test F1: {'mean': 0.6335342861658652, 'std': 0.015067839728839275, 'values': [0.6190476190476191, 0.6324324324324324, 0.6491228070175439]}


## 7. Verify Canonical Artifacts

The repository already lives in Drive, and every run writes directly to the canonical `artifacts/` directory. No post-run copying is needed.

In [7]:
from pathlib import Path

artifact_dir = Path(PROJECT_DIR) / 'artifacts'
required = [
    'baseline_metrics.json',
    'lstm_metrics.json',
    'experiment_summary.json',
    'frozen_lstm_config.json',
    'best_model.pt',
]
missing = [name for name in required if not (artifact_dir / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing canonical artifacts: {missing}')
print(f'Canonical evidence verified in {artifact_dir}')

Canonical evidence verified in /content/drive/MyDrive/APS360/Final_Project/artifacts
